# [Anthropic API](https://platform.claude.com/docs/en/home)

In [ ]:
# Initialize the Anthropic client

from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"
# model = "claude-sonnet-4-6"
max_tokens = 100

In [ ]:
# Send a message to the model and receive a response

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{"role": "user", "content": "Hello Claude, how are you?"}],
)

# Display the main content
print(f"Response: {message.content[0].text}")
print(f"Model: {message.model}")
print(f"Stop Reason: {message.stop_reason}")
print(f"Role: {message.role}")
print("Usage:")
print(f"  Input tokens: {message.usage.input_tokens}")
print(f"  Output tokens: {message.usage.output_tokens}")

## [Context Management](https://platform.claude.com/docs/en/build-with-claude/working-with-messages) | Math-tutor

### Selection Probability

Before generating each token (word), the model calculates a probability for every possible next word based on the context:

```
Input: "What do you think"
Next token probabilities:
  about  → 30%
  would  → 20%
  of     → 10%
  is     → 10%
  when   → 10%
  makes  → 5%
  we     → 5%
```

The model then **selects one token** based on these probabilities. A higher probability = more likely to be chosen.

### Temperature: Controlling Randomness

**Temperature** is a parameter that adjusts these selection probabilities. Using our example above:

| Range | Level | Effect on Example | Use Cases |
|---|---|---|---|
| **0.0 - 0.3** | **Low** | `about → 100%`, others → 0% | Factual responses, coding, data extraction, content moderation |
| **0.4 - 0.7** | **Medium** | `about → 30%`, `would → 20%`, `of → 10%` (unchanged) | Summarization, educational content, problem-solving, creative writing with constraints |
| **0.8 - 1.0+** | **High** | `about → 23%`, `would → 18%`, `of → 15%` (flattened distribution) | Brainstorming, creative writing, marketing, joke generation |

**Key principle:** Use lower temperatures when accuracy and consistency matter; use higher temperatures when diversity and creativity matter.

**In this tutor:** `temperature=1.0` keeps responses natural and varied without becoming nonsensical.

In [ ]:
# Example of a math tutor system prompt


def creative_tutor_math(messages: list[dict[str, str]]) -> str:
    """Generate a tutoring response using the math tutor system prompt.

    Args:
        messages: List of message dictionaries with 'role' and 'content' keys.

    Returns:
        str: The assistant's tutoring response.
    """
    system_prompt = """
        You are a patient math tutor.
        Do not directly answer a student's questions.
        Guide them to a solution step by step.
    """
    temperature = 0.0  # Adjust creativity level

    message = client.messages.create(
        model=model,
        max_tokens=200,
        messages=messages,
        system=system_prompt,
        temperature=temperature,
    )
    return message.content[0].text


messages: list[dict[str, str]] = []

# Use a 'while True' loop to run the chatbot forever, press ESC to stop
while True:
    try:
        # Get user input
        user_input = input("> ")
        print(">", user_input)

        # Add user input to the list of messages
        messages.append({"role": "user", "content": user_input})

        # Call Claude with the 'creative_tutor_math' function
        response = creative_tutor_math(messages)
        display(Markdown(response))

        # Add generated text to the list of messages
        messages.append({"role": "assistant", "content": response})
    except Exception as e:
        if "400" in str(e):
            break
        raise

## [Streaming Messages](https://platform.claude.com/docs/en/build-with-claude/streaming)

When streaming is enabled, the API sends multiple events instead of a single response. Each event type represents a step in the message generation:

| Event | Description |
|-------|-------------|
| **message_start** | Contains a Message object with empty content. Emitted once at the start of streaming. |
| **content_block_start** | Marks the beginning of a new content block (text, tool use, etc.). |
| **content_block_delta** | Contains a chunk of generated text or tool data for the current content block. Multiple deltas are sent as the block is generated. |
| **content_block_delta** |  |
| **content_block_stop** | Marks the end of the current content block. |
| **message_delta** | Contains top-level changes to the Message (e.g., final stop reason, token usage updates). |
| **message_stop** | Marks the end of the message stream. The complete message is now available. |

In [ ]:
# Example of streaming responses from the model

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Hello Claude, how are you? What is the weather like today?"}
    ],
    temperature=1.0,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

## [Structured Output](https://platform.claude.com/docs/en/test-and-evaluate/develop-tests#grade-your-evaluationsz)

In [12]:
# Define a tool with input schema
response = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[
        {
            "role": "user",
            "content": (
                "Extract the key information from this email: John Smith (john@example.com)\n"
                "is interested in our Enterprise plan and wants to schedule a demo for next Tuesday at 2pm."
            ),
        }
    ],
    # Output schema - defines structure of Claude's response
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "email": {"type": "string"},
                    "plan_interest": {"type": "string"},
                    "demo_requested": {"type": "boolean"},
                },
                "required": ["name", "email", "plan_interest", "demo_requested"],
                "additionalProperties": False,
            },
        }
    },
)

response

"""
Also, try this command in terminal

claude -p "Generate a random user" \
  --output-format json \
  --json-schema '{"type":"object","properties":{"email":{"type":"string"}},"required":["email"]}'
"""

'\nAlso, try this command in terminal\n\nclaude -p "Generate a random user"   --output-format json   --json-schema \'{"type":"object","properties":{"email":{"type":"string"}},"required":["email"]}\'\n'

In [ ]:
# Tool Calls Example with structured data parsing

import json

tools = [
    {
        "name": "extract_user_data",
        "description": "Extract structured user information from text",
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer"},
                "email": {"type": "string"},
                "interests": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["name", "age", "email"],
        },
    }
]

# --- Turn 1: Get structured data from Claude ---
response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "tool", "name": "extract_user_data"},
    messages=[
        {
            "role": "user",
            "content": "Extract info: John Doe is 28, reach him at john@example.com. He loves hiking and photography.",
        }
    ],
)

# Extract the tool_use block
tool_use_block = next(b for b in response.content if b.type == "tool_use")
structured_data = tool_use_block.input  # ✅ Your guaranteed JSON
print("Extracted:", json.dumps(structured_data, indent=2))

# --- Turn 2: Send tool_result back (required to continue conversation) ---
followup = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    tools=tools,
    messages=[
        {
            "role": "user",
            "content": "Extract info: John Doe is 28, reach him at john@example.com. He loves hiking.",
        },
        {
            "role": "assistant",
            "content": response.content,  # Include Claude's full response (with tool_use)
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,  # Must match the tool_use id
                    "content": "Data saved successfully.",  # Acknowledge the tool call
                }
            ],
        },
    ],
)

print("Claude continues:", followup.content)

In [ ]:
# Assistant Prefilling Example of structured data generation and parsing

import json

# Structured data

response = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {"content": "Generate a random user data in json format", "role": "user"},
        {
            "content": "Here is it, just one user, with fields: user_id, name, email, and phone. Nothing more...\n```json",
            "role": "assistant",
        },
    ],
    stop_sequences=["```"],
)

print("AI message:", response.content[0].text)

event_bridge_rule = json.loads(response.content[0].text)
print("Json parsed:", event_bridge_rule)

## [Prompt evaluation](https://platform.claude.com/docs/en/test-and-evaluate/develop-tests#grade-your-evaluations) & [Prompt engineering](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview)

Graders:
- Code
- Model
- Human


In [ ]:
# Test cases for grading

python_criteria = """
The function must satisfy ALL of the following:
    - Uses a single list comprehension with a single compound condition (no nested comprehensions or helper functions).
    - Accesses the state via `instance.get('State') == 'running'` (flat string value, NOT a nested dict like `instance.get('State', {}).get('Name')`).
    - Uses `instance.get('InstanceType', '').startswith(('t2', 't3'))` with a TUPLE argument (not two separate startswith calls).
    - Includes proper type hints: parameter annotated as `list[dict[str, Any]]`, return type as `list[dict[str, Any]]`.
    - Imports `Any` from the `typing` module.
    - Function name must be short but meaningful.
"""

json_criteria = """
The JSON policy must satisfy ALL of the following:
    1. Contains exactly THREE Statement entries.
    2. Statement 1 (Sid=AllowGetObject): Effect Allow, s3:GetObject, IpAddress condition 192.168.1.0/24.
    3. Statement 2 (Sid=DenyGetObjectOtherIPs): Effect Deny, s3:GetObject, NotIpAddress condition 192.168.1.0/24.
    4. Statement 3 (Sid=DenyInsecureTransport): Effect Deny, s3:GetObject, Condition Bool aws:SecureTransport=false — enforces HTTPS.
    5. Principal in all statements must be in object form {AWS: *}, NOT bare string *.
    6. Resource in all statements must be a JSON array containing both the bucket ARN (arn:aws:s3:::my-secure-bucket)
        AND the objects ARN (arn:aws:s3:::my-secure-bucket/*).
    7. Uses Version 2012-10-17.
"""

test_cases = [
    {
        "task": """
        Write a Python function that takes a list of EC2 instance dictionaries
        and returns only the instances where the state is 'running' and the instance type starts with 't2' or 't3'.
    """,
        "structured_output": """The output must be a pure Python function definition with no additional text
        or explanation. And without ``` md output fences.""",
        "format": "python",
        "criteria": python_criteria,
    },
    # Prompt engineered version
    {
        "task": """
        Write a Python function that takes a list of EC2 instance dictionaries
        and returns only the instances where the state is 'running' and the instance type starts with 't2' or 't3'.

        Guidance:
        - Include proper type hints: parameter as list[dict[str, Any]],
        - Use a single list comprehension to filter the instances based on the specified conditions.
        - Your output must be a single function definition with docstring or comments.
        - Output must be compileable Python code.

        Follow these steps:
        1. Define the function with the meaningful but short name.
        2. Access the state with instance.get('State').
        3. Check instance type with instance.get('InstanceType').

        Output example:
        ```python
        from typing import Any

        def func(instances: list[dict[str, Any]]) -> list[dict[str, Any]]:
            return [State == 'running' and InstanceType.startswith(('t2', 't3'))]
        ```
    """,
        "structured_output": """The output must be a pure Python function definition with no additional text
        or explanation. And without ``` md output fences.""",
        "format": "python",
        "criteria": python_criteria,
    },
    #   {
    #     "task": f"""
    #         Write a JSON object representing an S3 bucket policy that allows only GetObject actions
    #         from a specific IP address range (192.168.1.0/24) on all objects within the bucket 'my-secure-bucket'.
    #     """,
    #     "structured_output": "The output must be a valid JSON object with no additional text or explanation. And without ``` json output fences.",
    #     "format": "json",
    #     "criteria": json_criteria,
    #   }
]


def grade_json(text: str) -> int:
    try:
        json.loads(text)
        return 10
    except json.JSONDecodeError:
        return 0


def grade_python_code(text: str) -> int:
    try:
        compile(text, "<string>", "exec")
        return 10
    except SyntaxError:
        return 0


def model_grading_function(response: str, test_case: dict) -> str:
    eval_prompt = f"""
You are an expert code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{response}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "score": A number between 1-10
- "reasoning": A concise explanation of your overall assessment, 10 tokens or less

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "score": number,
    "reasoning": string
}}
    """

    grade_response = client.messages.create(
        model=model,
        max_tokens=100,
        messages=[
            {
                "content": f"""Evaluate the following response for the task: {test_case["task"]}
                Response: {response}\nUse the criteria: {test_case["criteria"]}""",
                "role": "user",
            },
            {"content": "```json", "role": "assistant"},
        ],
        stop_sequences=["```"],
        temperature=0.0,
        system=eval_prompt,
    )
    return json.loads(grade_response.content[0].text.strip())


for test_case in test_cases:
    response = client.messages.create(
        model=model,
        max_tokens=300,
        messages=[
            {"content": test_case["task"], "role": "user"},
            {"content": test_case["structured_output"], "role": "user"},
            # {"content": "Here is the solution:\n```", "role": "assistant"},
        ],
    )

    code_score = 0  # between 0-10 | 10 true, 0 false
    if test_case["format"] == "json":
        code_score = grade_json(response.content[0].text)
    elif test_case["format"] == "python":
        code_score = grade_python_code(response.content[0].text)

    model_grade = model_grading_function(response.content[0].text, test_case)

    score = (model_grade["score"] + code_score) / 2

    print(f"Test Case: {test_case['task']}")
    print(f"Code Score: {code_score}")
    print(f"Model Score: {model_grade['score']} - {model_grade['reasoning']}")
    print(f"Score: {score}")
    print(f"Response: \n{response.content[0].text}")
    print("-" * 50)
    print()

## [Tool Design](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)

### [Stop reasons](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#stop-sequence):
- **"tool_use"** - The model stopped because it decided to use a tool. The message content will include a ToolUseBlock with the details of the tool call.
- **"max_tokens"** - The model stopped because it reached the maximum token limit for the response.
- **"end_turn"** - The model stopped because it reached the end of its turn.
- **"stop_sequence"** - The model stopped because it encountered a stop sequence. Example `stop_sequences=["```"]` we used before.

In [ ]:
# Example of a failing test case


def sum(a: float, b: float) -> float:
    if (a is None) or (b is None):
        raise ValueError("Both a and b must be provided")

    return a + b


# Define the sum tool schema
sum_tool = {
    "name": "sum",
    "description": "Add two numbers together and return the result.",
    "input_schema": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "The first number."},
            "b": {"type": "number", "description": "The second number."},
        },
        "required": ["a", "b"],
    },
}

# Step 1: Send message with tool definition
response = client.messages.create(
    model=model,
    max_tokens=200,
    tools=[sum_tool],
    messages=[{"role": "user", "content": "What is 42.5 + 17.3?"}],
)

print(f"Stop reason: {response.stop_reason}")

# Step 2: Extract tool use block and call the actual function
tool_use_block = next(b for b in response.content if b.type == "tool_use")
tool_input = tool_use_block.input
print(f"Tool called: {tool_use_block.name}({tool_input})")

result = sum(**tool_input)
print(f"Tool result: {result}")

# Step 3: Return tool result back to Anthropic
final_response = client.messages.create(
    model=model,
    max_tokens=200,
    tools=[sum_tool],
    messages=[
        {"role": "user", "content": "What is 42.5 + 17.3?"},
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": str(result),
                }
            ],
        },
    ],
)

print(f"\nFinal response: {final_response.content[0].text}")

## RAG (Retrieval Augmented Generation)

1. Data -> Chunking -> Embedding -> Normalization -> Vector DB
2. User query -> Embedding -> Vector DB -> [Cosine similarity] -> Response

> Cosine Similarity: cos(A, B) = (A · B) / (||A|| * ||B||)
> Cosine Distance: 1 - cos(A, B)

### Chunking strategy:
- **Size Based** - Split text into fixed-size chunks (e.g., 500 tokens) without considering content boundaries.
- **Structure Based** - Split text based on natural boundaries (e.g., paragraphs, sections) to preserve context.
- **Semantic Based** - Use embeddings to group semantically similar sentences or paragraphs into chunks, regardless of their position in the text.

> Semantic Search vs Lexical Search
> 
> The main difference between semantic search and lexical search is that semantic search understands the meaning and context of the query, while lexical search matches keywords without understanding their meaning. Semantic search uses techniques like word embeddings to capture the relationships between words, allowing it to return relevant results even if they don't contain the exact keywords used in the query. Lexical search, on the other hand, relies on exact keyword matches, which can lead to less relevant results if the query and documents use different wording.

Also, there is a **Reciprocal Rank Fusion** - a method that combines the results of multiple search algorithms by assigning a score to each document based on its rank in each algorithm's results. The score is calculated as the reciprocal of the rank (1/rank), and the final score for each document is the sum of its scores across all algorithms. This approach helps to improve search relevance by leveraging the strengths of different algorithms and mitigating their weaknesses.

In [ ]:
import voyageai

voyageai_client = voyageai.Client()

# Size Based chunks...
chunks = [
    "This is the first buggy-chunk.",
    "This is the second flaky-chunk.",
    "This is the third hazy-chunk.",
]

result = voyageai_client.embed(chunks, model="voyage-2")
chunks_embeddings = result.embeddings
print("Embedding result:", chunks_embeddings)

In [ ]:
from vector_index import VectorIndex

vector_index = VectorIndex()

for i, chunk in enumerate(chunks):
    vector_index.add_vector(chunks_embeddings[i], {"content": chunk})

user_embedding = voyageai_client.embed(["2 hazy"], model="voyage-2").embeddings[0]

similar_chunks = vector_index.search(user_embedding)

print("Similar chunks:", similar_chunks)

In [ ]:
from bm25_index import BM25Index

bm25_index = BM25Index()

for chunk in chunks:
    bm25_index.add_document({"content": chunk})

for doc, distance in bm25_index.search("2 hazy", 3):
    print(f"Document: {doc}, Distance: {distance}")

## Batch

In [ ]:
batch = client.messages.batches.create(
    requests=[
        {
            "custom_id": "task-sentiment-positive",
            "params": {
                "model": model,
                "max_tokens": max_tokens,
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Is this review positive or negative? Reply with one word only.\n"
                            "Review: 'This product changed my life, absolutely love it!'"
                        ),
                    }
                ],
            },
        },
        {
            "custom_id": "task-sentiment-negative",
            "params": {
                "model": model,
                "max_tokens": max_tokens,
                "messages": [
                    {
                        "role": "user",
                        "content": (
                            "Is this review positive or negative? Reply with one word only.\n"
                            "Review: 'Terrible quality, broke after one day. Waste of money.'"
                        ),
                    }
                ],
            },
        },
    ]
)

batch

MessageBatch(id='msgbatch_01SAaAY3jfGMJnf7X9jjfDw5', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 4, 10, 11, 58, 19, 75768, tzinfo=datetime.timezone.utc), ended_at=None, expires_at=datetime.datetime(2026, 4, 11, 11, 58, 19, 75768, tzinfo=datetime.timezone.utc), processing_status='in_progress', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0), results_url=None, type='message_batch')

In [ ]:
client.messages.batches.retrieve(batch.id)

In [ ]:
client.messages.batches.results(batch.id)